In [ ]:
!pip install huggingface_hub

In [ ]:
clip_storage = "/workspaces/microns_phase3_nda/fetching_stimuli/temp_clip_storage"
dir_in_repo='stimuli/raw_microns'
repo_id='SamuelSuhai/visual_codes_age'

In [ ]:
import datajoint as dj
import sys
import os
sys.path.append("/workspaces/microns_phase3_nda/")
os.makedirs(clip_storage, exist_ok=True)

from microns_phase3 import nda, utils


from huggingface_hub import HfApi, login
import numpy as np
import json

In [ ]:
login()
api = HfApi() 

In [ ]:
nda.Clip()

In [ ]:

def save_all_clips(clip_storage, non_clip_fields):
    for clip_num, key in enumerate(nda.Clip().fetch("condition_hash")):
        tab = nda.Clip & dict(condition_hash=key)
        clip = tab.fetch1("clip")
        meta = tab.fetch1(*non_clip_fields, as_dict=True)
        np.save(os.path.join(clip_storage, f'{clip_num}.npy'), clip)
        with open(os.path.join(clip_storage, f'{clip_num}_meta.json'), 'w') as f:
            json.dump(meta, f)
        print(f"Saved clip {clip_num}")




In [ ]:
non_clip_fields = [f for f in nda.Clip.heading.names if f != 'clip']
save_all_clips(clip_storage, non_clip_fields)

api.upload_folder(
    folder_path=clip_storage,
    repo_id=repo_id,
    repo_type='dataset',
    path_in_repo=dir_in_repo
)
